# 🔴 NASA Space Apps: Mars EVA Tactical Co-Pilot Fine-Tuning & Quantization
### 100% Authentic NASA PDS & USGS Dataset -> 4-Bit Quantized GGUF Edge Model

This notebook trains **Qwen2.5-1.5B-Instruct** on **5,800+ authentic NASA PDS CRISM/OMEGA records, USGS Planetary Gazetteer entries, and NASA HLS2 rover mobility guidelines**.

- **Runtime:** Free Google Colab T4 GPU (Menu: Runtime -> Change runtime type -> T4 GPU)
- **Duration:** ~15 to 20 minutes
- **Output:** `mars_eva_q8.gguf` ready for 100% offline edge deployment on a Mars rover/suit.

In [1]:
# Step 1: Install standard HuggingFace & PyTorch dependencies (~25 seconds)
# Note: Standard pre-built PyPI wheels only (no git compilation to prevent hanging)
print("[1/7] Installing dependencies from PyPI...")
!pip install -q -U transformers peft trl bitsandbytes datasets accelerate
print("[1/7] Installation complete!")

[1/7] Installing dependencies from PyPI...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 14.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
[1/7] Installation complete!


In [2]:
# Step 2: Upload your authentic NASA training dataset
# Upload 'nasa_mars_authentic_train.jsonl' from your project's packages/data folder
from google.colab import files
import os

if not os.path.exists('nasa_mars_authentic_train.jsonl'):
    print("[2/7] Please select and upload 'nasa_mars_authentic_train.jsonl'...")
    uploaded = files.upload()
else:
    print("[2/7] Found existing 'nasa_mars_authentic_train.jsonl'!")

[2/7] Found existing 'nasa_mars_authentic_train.jsonl'!


In [3]:
# Step 3: Load base model in 4-bit with BitsAndBytes (~1 minute)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"[3/7] Loading {model_id} in 4-bit quantization...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

# LoRA adapter configuration targeting attention and MLP projections
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("[3/7] Model and LoRA configuration ready!")

[3/7] Loading Qwen/Qwen2.5-1.5B-Instruct in 4-bit quantization...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
[3/7] Model and LoRA configuration ready!


In [4]:
# Step 4: Tokenize authentic NASA dataset (~30 seconds)
from datasets import load_dataset

print("[4/7] Loading NASA dataset...")
dataset = load_dataset('json', data_files='nasa_mars_authentic_train.jsonl', split='train')
print(f"Loaded {len(dataset)} verified NASA training conversations.")

def format_prompts(batch):
    texts = []
    for msgs in batch['messages']:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {'text': texts}

dataset = dataset.map(format_prompts, batched=True)
print("[4/7] Dataset tokenization ready!")

[4/7] Loading NASA dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Loaded 5824 verified NASA training conversations.


Map:   0%|          | 0/5824 [00:00<?, ? examples/s]

[4/7] Dataset tokenization ready!


In [14]:
# Step 5: Fine-tune on NASA Mars data (~15 minutes on T4 GPU)
import gc
import torch
from trl import SFTTrainer

print("[5/7] Starting QLoRA fine-tuning run on authentic NASA data...")

try:
    # Try the most modern TRL Config setup (>= v0.12.0)
    from trl import SFTConfig
    training_args = SFTConfig(
        output_dir="./mars_eva_adapter",
        dataset_text_field="text",
        max_seq_length=1024,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=False,        # Disable FP16 scaling to prevent GradScaler BFloat16 conflicts
        bf16=False,        # Disable BF16 since T4 doesn't support it natively
        logging_steps=20,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        report_to="none"
    )
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        args=training_args,
        processing_class=tokenizer
    )
except Exception as e:
    print(f"Modern SFTConfig setup failed ({type(e).__name__}). Applying robust legacy fallback...")
    from transformers import TrainingArguments

    training_args = TrainingArguments(
        output_dir="./mars_eva_adapter",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=False,        # Disable FP16 scaling to prevent GradScaler BFloat16 conflicts
        bf16=False,        # Disable BF16 since T4 doesn't support it natively
        logging_steps=20,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        report_to="none"
    )

    # Resolve kwargs dynamically based on older trl implementation versions
    sft_kwargs = {
        "model": model,
        "train_dataset": dataset,
        "args": training_args
    }

    # Try to set dataset text formatting arguments dynamically
    import inspect
    sig = inspect.signature(SFTTrainer.__init__)

    if "dataset_text_field" in sig.parameters:
        sft_kwargs["dataset_text_field"] = "text"
    if "max_seq_length" in sig.parameters:
        sft_kwargs["max_seq_length"] = 1024

    # Resolve tokenizer argument name dynamically
    if "processing_class" in sig.parameters:
        sft_kwargs["processing_class"] = tokenizer
    else:
        sft_kwargs["tokenizer"] = tokenizer

    trainer = SFTTrainer(**sft_kwargs)

trainer.train()
model.save_pretrained("./mars_eva_adapter")
tokenizer.save_pretrained("./mars_eva_adapter")
print("[5/7] Fine-tuning complete! LoRA adapter saved to ./mars_eva_adapter")

[5/7] Starting QLoRA fine-tuning run on authentic NASA data...
Modern SFTConfig setup failed (TypeError). Applying robust legacy fallback...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,2.468035
40,0.337731
60,0.228980
80,0.209039
100,0.213746
120,0.222495
140,0.202244
160,0.210390
180,0.227195
200,0.212058


[5/7] Fine-tuning complete! LoRA adapter saved to ./mars_eva_adapter


In [16]:
# Step 6: Merge LoRA weights into standalone 16-bit model (~1 minute)
# Update torchao first to satisfy PEFT's compatibility constraint
!pip install -q -U torchao

import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import shutil

print("[6/7] Merging LoRA weights into standalone base model...")

# Free GPU and trainer memory to ensure smooth merge on Colab RAM
try:
    del trainer
    del model
except NameError:
    pass

torch.cuda.empty_cache()
gc.collect()

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True
)

merged_model = PeftModel.from_pretrained(base_model, "./mars_eva_adapter")
merged_model = merged_model.merge_and_unload()

merged_dir = "./mars_eva_merged"
merged_model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print("[6/7] Merged model saved to ./mars_eva_merged")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 32.7 MB/s eta 0:00:00
[6/7] Merging LoRA weights into standalone base model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[6/7] Merged model saved to ./mars_eva_merged


In [18]:
# Step 7: Convert to GGUF format for 100% offline edge inference (~2 minutes)
import os
import json

print("[7/7] Converting to GGUF format using llama.cpp...")
if not os.path.exists("llama.cpp"):
    !git clone --depth=1 https://github.com/ggml-org/llama.cpp.git
!pip install -q -r llama.cpp/requirements.txt
!pip install -q gguf protobuf

# Sanitize tokenizer_config.json to prevent AttributeError during conversion
tok_config_path = "./mars_eva_merged/tokenizer_config.json"
if os.path.exists(tok_config_path):
    print("Sanitizing tokenizer configuration to maintain llama.cpp compatibility...")
    with open(tok_config_path, "r", encoding="utf-8") as f:
        config_data = json.load(f)

    # If extra_special_tokens is a list (which causes .keys() to crash), remove or replace it
    if "extra_special_tokens" in config_data:
        if isinstance(config_data["extra_special_tokens"], list):
            config_data["extra_special_tokens"] = {}

    with open(tok_config_path, "w", encoding="utf-8") as f:
        json.dump(config_data, f, indent=2, ensure_ascii=False)

# Convert to 8-bit quantized GGUF directly (q8_0 is ~1.5GB, fast & high fidelity)
!python llama.cpp/convert_hf_to_gguf.py ./mars_eva_merged --outfile ./mars_eva_q8.gguf --outtype q8_0

print("GGUF Export complete! File: mars_eva_q8.gguf")

# Automatically trigger browser download to your laptop
from google.colab import files
if os.path.exists('mars_eva_q8.gguf'):
    print('Downloading mars_eva_q8.gguf to your computer...')
    files.download('mars_eva_q8.gguf')
else:
    print('Export failed. Check output above.')

[7/7] Converting to GGUF format using llama.cpp...
Sanitizing tokenizer configuration to maintain llama.cpp compatibility...
INFO:hf-to-gguf:Loading model: mars_eva_merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> Q8_0, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> Q8_0, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> Q8_0, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> Q8_0, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gg

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>